# Read a Parquet file in Python with chDB (drop-in pandas)

Companion to [read-parquet-file-python](https://clickhouse.com/resources/engineering/read-parquet-file-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas pyarrow`.

In [ ]:
import pandas as _real_pd
import chdb.datastore as pd

_real_pd.set_option("display.float_format", "{:.2f}".format)

## 1. Read a Parquet file into a DataFrame (schema from metadata)

In [ ]:
df = pd.read_parquet("data/events.parquet")
df

In [ ]:
df.dtypes

## 2. Filter + aggregate the way you already do (pandas, not SQL)

In [ ]:
revenue = (df[df["event_type"] == "purchase"]
           .groupby("country")["amount"].sum()
           .sort_values(ascending=False))
revenue

## 3. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

## 4. Performance: same code, one import swapped, on a 20M-row Parquet file

Apple M4 Pro (14 cores, 24 GB RAM, macOS), best-of-3 warm: ~3x faster than `pandas.read_parquet`.

In [ ]:
import time

def datastore_agg():
    d = pd.read_parquet("data/events.parquet")
    r = (d[d["event_type"] == "purchase"]
         .groupby("country")["amount"].sum()
         .sort_values(ascending=False))
    return r.to_pandas()

def pandas_agg():
    import pandas as real_pd
    p = real_pd.read_parquet("data/events.parquet")
    return (p[p["event_type"] == "purchase"]
            .groupby("country")["amount"].sum()
            .sort_values(ascending=False))

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

pd_s = best_of_3(pandas_agg)
ds_s = best_of_3(datastore_agg)
print(f"import pandas as pd:            {pd_s:.3f}s")
print(f"import chdb.datastore as pd:    {ds_s:.3f}s")
print(f"speedup:                        {pd_s / ds_s:.1f}x")